In [ ]:
import os
from os import path
import pickle
import numpy as np
from tqdm import tqdm

from macaquethings.plotting.default_styles import *
from macaquethings.plotting.util import legend_without_duplicate_labels
from macaquethings.data_util.load_data import get_channel_masks

import matplotlib.pyplot as plt
import seaborn as sns

figure_style()  # set consistent plotting defaults for all figs
%matplotlib widget

In [ ]:
# specify general options
monkey = 'monkeyN'  # use for the whole script


arrays = list(range(1, 17))
if monkey == 'monkeyN':
    arrays.remove(6)

array_str = "_".join([str(a) for a in arrays])
sessions = "0_3_4_5" if monkey == "monkeyN" else "0_1_2_3_4_5"  # all sessions

roi_to_color = {
    'v1': 'tab:blue',
    'v4': 'tab:orange',
    'it': 'tab:green'
}

# decoding_dir = path.join('..', '..', 'results', 'decoding', f'{monkey}_stratifiedshuffle_allsess_category_vs_image')
decoding_dir = path.join('..', '..', 'results', 'decoding', f'{monkey}_stratifyimages_allsess')


In [ ]:
rois = [1, 2, 3]

label_info = []
roi_info = []
roi_decoding_pkls = []

for roi in rois:
    fname_cat = f"{monkey}-labels_category-sessions_{sessions}-rois_{roi}-arrays_{array_str}-baseline_0-standardize_1-lfp.pkl"
    
    roi_decoding_pkls.append(fname_cat)
    
    label_info.append('category')
    
    roi_info.append(
        {1: 'v1', 2: 'v4', 3: 'it'}[roi]
    )
    
print("loading roi decoding results:")
roi_decoding_results = []
for decoding_pkl in tqdm(roi_decoding_pkls):
    try:
        with open(path.join(decoding_dir, decoding_pkl), 'rb') as f:
            res = pickle.load(f)
            t = res['times']
            acc = res['accuracies']
            roi_decoding_results.append({
                'time': t,
                'accuracy': acc
            })
    except Exception as e:
        print(e)
        roi_decoding_results.append(
            {
                'time': [],
                'accuracy': []
            }
        )


# load oracle corr data

In [ ]:
label = 'filenames'  # category or filenames

# NOTE ON ARRAYS
# contrary to all other analyses, because each channel is independent here oracle correlation is computed for all electrodes in all arrays
# for consistency with later analyses the plots generated here explicitly exclude array 6, since it is reported to be broken and is also excluded
# in other analyses

cfg_overrides = {
    'good_channel_threshold': 1.5  # oracle corrs were computed for all channels, apply same threshold as for other analyses
}

# --- derived from above
# oracle_dir = path.join('..', '..', 'results', 'oracle_electrode', f'{monkey}_lfp')
oracle_dir = path.join('..', '..', 'results', 'oracle_electrode', f'{monkey}')
oracle_file = f'{monkey}-labels_{label}-sessions_{sessions}-rois_1_2_3-arrays_1_2_3_4_5_6_7_8_9_10_11_12_13_14_15_16-baseline_0-standardize_1.pkl'

with open(path.join(oracle_dir, oracle_file), 'rb') as f:
    oracle_results = pickle.load(f)

oracle_corr = oracle_results['oracle'].T  # for this script convention is to have time first
oracle_time = oracle_results['time']

data_cfg = oracle_results["data_cfg"]
data_cfg.update(cfg_overrides)
oracle_masks = get_channel_masks(data_cfg, root='../..')

oracle_rois = oracle_masks['rois']
oracle_arrays = oracle_masks['array_ids']
good_channels = oracle_masks['good_channels']

# if monkeyN, make sure we exclude all channels for array 6
if monkey == 'monkeyN':
    good_channels[oracle_arrays == 6] = False

In [ ]:
savedir = "decoding_oracle_panels"
os.makedirs(savedir, exist_ok=True)

xlim = (0, 300)
# FIG 1 --- all ROIs
fig, ax1 = plt.subplots(1, 1, figsize=(THIRD_WIDTH, THIRD_WIDTH / 2))

# --------------- DECODING
for roi, label, res in zip(roi_info, label_info, roi_decoding_results):
    if not roi == 'it':
        continue
    ax1.plot(res['time'], res['accuracy'], color='tab:green', zorder=3, linewidth=1.)
ax1.set_xlabel('time (ms)')
ax1.set_ylabel('decoding accuracy', color='tab:green')
ax1.set_title(f"Decoding and signal reliability - IT, {monkey}")
plt.xlim(xlim)
ax1.tick_params(axis='y', labelcolor='tab:green')

# --------------- ORACLE
ax2 = ax1.twinx()
roi, roiname = 3, 'IT'
mask = (oracle_rois == roi) & good_channels
oracles_select = oracle_corr[:, mask]
ax2.plot(oracle_time, oracles_select, color='tab:purple', alpha=.01, linewidth=1)
ax2.plot(oracle_time, oracles_select.mean(axis=1), color='tab:purple', alpha=1., linewidth=1.)
ax2.set_ylabel('oracle correlation', color='tab:purple')
ax2.tick_params(axis='y', labelcolor='tab:purple')
plt.savefig(path.join('decoding_oracle_panels', f'{monkey}_oracle_and_decoding.svg'))
